In [9]:
ppt_output = Path('docs/AI_IT_Support_Architecture.pptx')
if not ppt_output.exists():
    raise FileNotFoundError(f'PowerPoint was not created at: {ppt_output}')

ppt_size = ppt_output.stat().st_size
print(f'PowerPoint exists: {ppt_output.exists()}')
print(f'PowerPoint size (bytes): {ppt_size}')
print(f'Slide count: {len(Presentation(str(ppt_output)).slides)}')
print('Notebook validation complete.')


PowerPoint exists: True
PowerPoint size (bytes): 36723
Slide count: 9
Notebook validation complete.


# Save and Verify `.pptx` Output

Save the presentation to disk and confirm the file exists with valid slide count.

In [8]:
from pptx.util import Pt

prs = Presentation()
prs.slide_width = 12192000
prs.slide_height = 6858000

# Title slide
slide = prs.slides.add_slide(prs.slide_layouts[0])
slide.shapes.title.text = 'AI IT Support Assistant Architecture'
slide.placeholders[1].text = 'Grounded support workflow, tool routing, and data architecture'

# Content slides
for item in slides:
    slide = prs.slides.add_slide(prs.slide_layouts[1])
    slide.shapes.title.text = item['title']
    body = slide.placeholders[1].text_frame
    body.clear()
    for idx, bullet in enumerate(item['bullets'][:6]):
        if idx == 0:
            p = body.paragraphs[0]
            p.text = bullet
        else:
            p = body.add_paragraph()
            p.text = bullet
        p.level = 0
        p.font.size = Pt(18)

# Add a closing summary slide
slide = prs.slides.add_slide(prs.slide_layouts[1])
slide.shapes.title.text = 'Key Takeaways'
body = slide.placeholders[1].text_frame
body.clear()
for i, text in enumerate([
    'Deterministic validation prevents invalid or duplicate ticket actions.',
    'Sentence-transformer routing improves semantic intent detection.',
    'Structured tool outputs keep responses grounded and user-friendly.',
    'The architecture is extensible for future access, approval, and lifecycle features.'
]):
    if i == 0:
        p = body.paragraphs[0]
        p.text = text
    else:
        p = body.add_paragraph()
        p.text = text
    p.level = 0
    p.font.size = Pt(18)

prs.save(ppt_output)
print(f'PowerPoint created: {ppt_output}')
print(f'Slides generated: {len(prs.slides)}')


PowerPoint created: docs\AI_IT_Support_Architecture.pptx
Slides generated: 9


# Generate PowerPoint with `python-pptx`

Create a presentation deck with a title slide and one slide per architecture section from the updated HTML content.

In [5]:
slides = []
for section in soup.select('section.card'):
    title = section.find('h2')
    if not title:
        continue
    bullets = []
    for text in section.select('p, li'):
        val = text.get_text(' ', strip=True)
        if val:
            bullets.append(val)
    slides.append({
        'title': title.get_text(strip=True),
        'bullets': bullets[:6]
    })

for idx, slide in enumerate(slides, 1):
    print(f'{idx}. {slide["title"]}: {len(slide["bullets"])} items')

print(f'Total slide-ready sections: {len(slides)}')


1. System Overview: 5 items
2. Runtime Components: 6 items
3. Request Flow: 5 items
4. Routing Strategy: 6 items
5. Safety and Validation: 4 items
6. Data Domain: 4 items
7. 16) How The Application Works (Updated): 1 items
Total slide-ready sections: 7


# Extract Structured Content for Slides

Normalise the page sections into a list of slide-ready entries so the PowerPoint deck mirrors the architecture content cleanly.

In [4]:
required_headings = ['System Overview', 'Runtime Components', 'Request Flow', 'Routing Strategy', 'Safety and Validation', 'Data Domain']
found_headings = [h.get_text(strip=True) for h in soup.select('h2')]
missing = [heading for heading in required_headings if heading not in found_headings]
if missing:
    raise ValueError(f'Missing required headings: {missing}')

updated_html = soup.prettify(formatter='html')
source_html.write_text(updated_html, encoding='utf-8')
print(f'Updated HTML written to: {source_html}')
print(f'Headings found: {len(found_headings)}')
print(f'Critical sections present: {len(required_headings)}')


Updated HTML written to: docs\architecture.html
Headings found: 7
Critical sections present: 6


# Validate and Write Updated HTML

Check the required headings and section count, then write the modified document back to disk.

In [3]:
soup = BeautifulSoup(original_html, 'html.parser')

# Update title and header metadata
if soup.title:
    soup.title.string = 'AI IT Support Assistant Architecture'

h1 = soup.find('h1')
if h1 is None:
    body = soup.body or soup
    h1 = soup.new_tag('h1')
    h1.string = 'AI IT Support Assistant'
    body.insert(0, h1)
else:
    h1.string = 'AI IT Support Assistant'

# Replace or inject a concise architecture summary section
main = soup.find('main')
if main is None:
    main = soup.new_tag('main')
    body = soup.body or soup
    body.append(main)

main.clear()
main.append(BeautifulSoup('''
  <section class="card">
    <h2>System Overview</h2>
    <p>The AI IT Support Assistant is a grounded support workflow for employee issues, ticket requests, knowledge lookup, and service health checks. It combines a Streamlit chat interface, a LangGraph orchestration layer, and a tool layer connected to PostgreSQL and knowledge content.</p>
    <ul>
      <li>Natural-language understanding for help requests and issue triage.</li>
      <li>Deterministic validation for ticket IDs, employee IDs, and pending confirmations.</li>
      <li>Semantic routing using sentence-transformer similarity before heuristic fallback.</li>
      <li>Structured responses with tool output rendered in a user-friendly format.</li>
    </ul>
  </section>
  <section class="card">
    <h2>Runtime Components</h2>
    <div class="two-col">
      <div>
        <ul>
          <li><strong>UI:</strong> Streamlit app in app.py</li>
          <li><strong>Service:</strong> ITSupportAgent in src/service.py</li>
          <li><strong>Workflow:</strong> LangGraph orchestration in src/agent_graph.py</li>
          <li><strong>Intent layer:</strong> semantic classifier and response logic in src/llm_router.py</li>
        </ul>
      </div>
      <div>
        <ul>
          <li><strong>Tools:</strong> employee, ticket, knowledge, and status access in src/tools.py</li>
          <li><strong>Database:</strong> PostgreSQL layer via src/database.py and src/models.py</li>
          <li><strong>Configuration:</strong> environment-driven model selection in src/config.py</li>
          <li><strong>State:</strong> multi-turn context and pending actions managed in src/state.py</li>
        </ul>
      </div>
    </div>
  </section>
  <section class="card">
    <h2>Request Flow</h2>
    <ol>
      <li>User enters a message in the Streamlit chat UI.</li>
      <li>The service layer forwards the request and context to the LangGraph workflow.</li>
      <li>Intent detection chooses between knowledge lookup, employee lookup, ticket actions, or status checks.</li>
      <li>Validation rules run before write operations to reduce duplicates and invalid updates.</li>
      <li>Structured outputs are returned to the UI and displayed for the user.</li>
    </ol>
    <div class="diagram">
      <pre>user message
  ↓
Streamlit UI
  ↓
ITSupportAgent.handle_message()
  ↓
LangGraph workflow
  ├─ capture context
  ├─ detect intent
  ├─ validate guardrails
  ├─ execute tool(s)
  └─ compose response
  ↓
PostgreSQL + knowledge base + status data
  ↓
Assistant result + structured tables</pre>
    </div>
  </section>
  <section class="card">
    <h2>Routing Strategy</h2>
    <p>Intent resolution follows a layered strategy: exact deterministic matches first, then semantic sentence-transformer similarity, then keyword-based heuristic fallback, and finally a safe small-talk response.</p>
    <ul>
      <li>knowledge_search</li>
      <li>employee_lookup</li>
      <li>ticket_lookup</li>
      <li>ticket_create</li>
      <li>ticket_update</li>
      <li>system_status</li>
      <li>small_talk</li>
    </ul>
  </section>
  <section class="card">
    <h2>Safety and Validation</h2>
    <ul>
      <li>Missing employee IDs and ticket IDs are confirmed before performing write actions.</li>
      <li>Duplicate ticket checks reduce repeated or redundant reports.</li>
      <li>Database validation ensures created and updated records align to schema requirements.</li>
      <li>Grounded responses avoid making up unavailable facts or unsupported actions.</li>
    </ul>
  </section>
  <section class="card">
    <h2>Data Domain</h2>
    <ul>
      <li>employees: workforce records and identifiers</li>
      <li>tickets: support cases, priorities, and workflow state</li>
      <li>knowledge_base: policy and troubleshooting data</li>
      <li>system_status: service health and operational conditions</li>
    </ul>
  </section>
''', 'html.parser'))

# Ensure the document includes a small modern styling block if missing
if not soup.find('style'):
    style = soup.new_tag('style')
    style.string = '''
      body { font-family: "Segoe UI", sans-serif; background: #f6f9fc; color: #14213d; margin: 0; }
      main { max-width: 1100px; margin: 0 auto; padding: 32px 20px 60px; display: grid; gap: 18px; }
      .card { background: white; border: 1px solid #dfeaf5; border-radius: 18px; padding: 22px; box-shadow: 0 12px 30px rgba(15, 23, 42, 0.06); }
      h1, h2, h3 { color: #0f172a; }
      .two-col { display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 18px; }
      .diagram { margin-top: 16px; background: #0f172a; border-radius: 12px; padding: 16px; }
      pre { margin: 0; color: #dfeeff; white-space: pre-wrap; font-family: Consolas, monospace; line-height: 1.6; }
      ul, ol { padding-left: 22px; }
      li { margin: 8px 0; }
    '''
    soup.head.insert(0, style)

print('Updated title and overview content successfully.')


Updated title and overview content successfully.


# Parse and Update HTML Content

Use BeautifulSoup selectors to update the title, section headings, and content blocks in a way that is easier to maintain than brittle string replacement.

In [2]:
from pathlib import Path
from bs4 import BeautifulSoup

source_html = Path('docs/architecture.html')
if not source_html.exists():
    raise FileNotFoundError(f'Missing source file: {source_html}')

original_html = source_html.read_text(encoding='utf-8')
backup_name = f"{source_html.stem}_backup_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.html"
backup_path = backup_dir / backup_name
shutil.copy2(source_html, backup_path)
print(f'Backed up original to: {backup_path}')


Backed up original to: docs\backups\architecture_backup_20260908_172741.html


# Load and Back Up `architecture.html`

Create a timestamped backup before the HTML is rewritten so the original file remains recoverable.

In [1]:
import shutil
from datetime import datetime
from pathlib import Path

try:
    import bs4
except Exception:
    !pip install beautifulsoup4
    from bs4 import BeautifulSoup

try:
    from pptx import Presentation
except Exception:
    !pip install python-pptx
    from pptx import Presentation

source_html = Path('docs/architecture.html')
backup_dir = Path('docs/backups')
backup_dir.mkdir(exist_ok=True)
ppt_output = Path('docs/AI_IT_Support_Architecture.pptx')
print(f'Source HTML: {source_html.resolve()}')
print(f'Backup dir: {backup_dir.resolve()}')
print(f'PowerPoint output: {ppt_output.resolve()}')



   ---------------------------------------- 2/2 [beautifulsoup4]

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 60.8 MB/s  0:00:00

   ---------------------------------------- 0/3 [XlsxWriter]
   ------------- -------------------------- 1/3 [lxml]
   -------------------------- ------------- 2/3 [python-pptx]
   -------------------------- ------------- 2/3 [python-pptx]
   ---------------------------------------- 3/3 [python-pptx]

Source HTML: E:\aiproject\capstone\docs\architecture.html
Backup dir: E:\aiproject\capstone\docs\backups
PowerPoint output: E:\aiproject\capstone\docs\AI_IT_Support_Architecture.pptx


# Set Up Paths and Dependencies

This notebook updates the architecture page and exports a presentation-ready PowerPoint deck from the same content.